In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import utils as rce_model

In [3]:
def construct_ozone_layer(z_levels):
    # Returns optical depth of ozone layer (for SW rad)
    # --- Constants ---
    M_o3 = 48e-3  # kg/mol
    M_air = 28.97e-3  # kg/mol
    rho0 = 1.225  # kg/m³ surface air density
    H = 8000.0  # scale height, m
    g = 9.81  # m/s²

    # --- Air density profile (exponential atmosphere) ---
    rho_air = rho0 * np.exp(-z_levels / H)  # kg/m³

    # --- Ozone mixing ratio profile (kg_o3 / kg_air) ---
    peak_o3_mmr = 8e-6 * (M_o3 / M_air)  # 8 ppmv → mass mixing ratio
    # multiply by 0.05 since UV is only 5% of solar rad
    # multiply by 2/3 since only UV B and C absorbed


    bottom_o3 = np.zeros(len(z_levels[z_levels < 17000]))
    slope_lower = peak_o3_mmr / 8000
    o3_lower = (z_levels[(z_levels >= 17000) & (z_levels < 25000)] - 17000) * slope_lower
    slope_upper = -peak_o3_mmr / 8000
    o3_upper = (
        z_levels[(z_levels >= 25000) & (z_levels < 33000)] - 25000
    ) * slope_upper + peak_o3_mmr
    top_o3 = np.zeros(len(z_levels[z_levels >= 33000]))

    mmr_o3 = np.concatenate([bottom_o3, o3_lower, o3_upper, top_o3])

    # --- Convert to number density (molecules/m³) for cross-section approach ---
    N_A = 6.022e23
    n_o3 = (mmr_o3 * rho_air / M_o3) * N_A  # molecules/m³

    # # --- Absorption cross section at 253.7 nm: 1.15e-17 cm² = 1.15e-21 m² ---
    # sigma = 1.15e-21   # m²/molecule

    # Solar irradiance fractions and representative cross-sections by band
    # sigma values in m²/molecule

    bands = {
        "UV-C (200-280nm)": {"f_solar": 0.01, "sigma": 1.0e-22},  # band average
        "UV-B (280-315nm)": {"f_solar": 0.015, "sigma": 5.0e-24},  # band average
        "UV-A (315-400nm)": {"f_solar": 0.065, "sigma": 0.0},
        "VIS  (400-700nm)": {"f_solar": 0.44, "sigma": 0.0},
        "NIR  (700nm+)   ": {"f_solar": 0.47, "sigma": 0.0},
    }

    sigma_eff = sum(b["f_solar"] * b["sigma"] for b in bands.values())
    # ≈ 1.075e-24 m²/molecule
    # sigma_eff ≈ 2.5e-23 m²/molecule  — much smaller than monochromatic UV value

    # --- Optical depth ---
    tau_dz = sigma_eff * n_o3 * np.diff(z_levels,prepend=0)
    tau = np.cumsum(tau_dz[::-1])[::-1]

    # # Check your column-integrated ozone amount
    # N_col = np.trapezoid(n_o3, z_levels)  # molecules/m²
    # N_col_DU = N_col / 2.687e20  # convert to Dobson Units
    # print(f"Ozone column: {N_col:.3e} molecules/m²")
    # print(f"Ozone column: {N_col_DU:.1f} DU")
    # # Expect ~300 DU for a realistic atmosphere
    return tau, mmr_o3

def net_solar_flux_at_all_levels(z_levels, toa_fsw):
    tau, _ = construct_ozone_layer(z_levels)
    transmis_dz = np.exp(-1.66 * tau)
    transmitted_solar_rad = np.append(transmis_dz * toa_fsw,toa_fsw)
    return transmitted_solar_rad


In [ ]:
tau1xCO2 = 1.42  # tau1xCO2 is surface optical thickness for 1xCO2 concentration
tau2xCO2 = 1.48  # tau2xCO2 is surface optical thickness for 2xCO2 concentration

p_ratio_bottom = 1.0  # bottom of model in p/p0 coordinates
p_ratio_top = 0.0  # top of model in p/p0 coordinates
del_p_ratio = 0.005  # thickness of model layer in p/p0 coordinates
p0 = 100000.0  # surface pressure (units Pa)
H = 8000.0  # scale height (units m)

Ts = 308.2  # surface temperature
Ta = 259.2  # atmospheric temperature (constant with height)

r = 1.66  # r is the diffusivity factor

del_t = 1.0  # units day

F_SW=-280

V0 = 5.0

num_levels = (  # get number of levels
    int((p_ratio_bottom - p_ratio_top) / del_p_ratio) + 1
)  # from layers that fit
p_ratio_levels = np.linspace(
    p_ratio_bottom, p_ratio_top, num_levels  # p/p0 at levels  # levels are at top and
)  # bottom of layers
z_levels = -np.log(p_ratio_levels[:-1]) * H  # height at levels
num_layers = num_levels - 1  # get number of layers
p_ratio_layers = (
    p_ratio_levels[:-1] + p_ratio_levels[1:]
) / 2.0  # p/p0 at layer centers
p_layers = p_ratio_layers * p0  # pressure at layer centers
del_p = del_p_ratio * p0  # layer thickness (units Pa)
z_layers = -np.log(p_ratio_layers) * H  # height at layer centers

In [11]:
Ta_layers = np.full(num_layers, Ta)

tau1xCO2_levels = tau1xCO2 * p_ratio_levels
tau2xCO2_levels = tau2xCO2 * p_ratio_levels

emissivity1xCO2_layers = np.full(num_layers, r * tau1xCO2 * del_p_ratio)
emissivity2xCO2_layers = np.full(num_layers, r * tau2xCO2 * del_p_ratio)

In [12]:
trans1xCO2_levels_to_levels = rce_model.transmission_to_all_levels(tau1xCO2_levels, r)
trans2xCO2_levels_to_levels = rce_model.transmission_to_all_levels(tau2xCO2_levels, r)

In [15]:
Ts_rad1xCO2, Ta_rad1xCO2_layers = rce_model.radiative_equilibrium_temperature(
    Ts = Ts,
    T_layers = Ta_layers,
    del_t = del_t,
    del_p=del_p,
    emissivity_layers=emissivity1xCO2_layers,
    trans_levels_to_levels=trans1xCO2_levels_to_levels,
    z_layers=z_layers,
    F_SW=F_SW,
)

Done!


In [22]:
s_rad1xCO2_layers = rce_model.dry_static_energy(Ta_rad1xCO2_layers, z_layers)
T2m_rad1xCO2 = rce_model.temperature_at_2m(Ta_rad1xCO2_layers[0], z_layers[0])
f_sh_rad1xCO2 = rce_model.sensible_heat_flux(Ts_rad1xCO2, T2m_rad1xCO2, V0)

In [23]:
dTa1dt_rad1xCO2_SH = g0 * f_sh_rad1xCO2 / (c_p * del_p)
dTa1dt_rad1xCO2_SH *= 60.0 * 60.0 * 24.0

NameError: name 'g0' is not defined